# Session 11: Advanced Retrieval with LangChain

## Learning Objectives:

- Understand and implement multiple retrieval strategies for RAG
- Compare naive, BM25, multi-query, parent-document, contextual compression, ensemble, and semantic chunking approaches
- Build RAG chains over a health and wellness knowledge base using LangChain and QDrant

In the following notebook, we'll explore various methods of advanced retrieval using LangChain!

We'll touch on:

- Naive Retrieval
- Best-Matching 25 (BM25)
- Multi-Query Retrieval
- Parent-Document Retrieval
- Contextual Compression (a.k.a. Rerank)
- Ensemble Retrieval
- Semantic chunking

We'll also discuss how these methods impact performance on our set of documents with a simple RAG chain.

There will be two breakout rooms:

- 🤝 Breakout Room Part #1
  - Task 1: Getting Dependencies!
  - Task 2: Data Collection and Preparation
  - Task 3: Setting Up QDrant!
  - Task 4-10: Retrieval Strategies
- 🤝 Breakout Room Part #2
  - Activity: Evaluate with Ragas

---

# 🤝 Breakout Room Part #1

## Task 1: Getting Dependencies!

We're going to need a few specific LangChain community packages, like OpenAI (for our [LLM](https://platform.openai.com/docs/models) and [Embedding Model](https://platform.openai.com/docs/guides/embeddings)) and Cohere (for our [Reranker](https://cohere.com/rerank)).

We'll also provide our OpenAI key, as well as our Cohere API key.

> NOTE: Create a `.env` file in this directory with `OPENAI_API_KEY` and `COHERE_API_KEY` to avoid being prompted each time.

In [1]:
import os
import getpass
from dotenv import load_dotenv

load_dotenv()

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API Key:")

In [2]:
if not os.environ.get("COHERE_API_KEY"):
    os.environ["COHERE_API_KEY"] = getpass.getpass("Cohere API Key:")

In [3]:
from uuid import uuid4

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_API_KEY"] = getpass.getpass("LangChain API Key:")

os.environ["LANGCHAIN_PROJECT"] = f"AIM - ADVANCED RETRIEVAL - {uuid4().hex[0:8]}"

## Task 2: Data Collection and Preparation

We'll be using our Health and Wellness Guide - a comprehensive resource covering exercise, nutrition, sleep, stress management, habits, and common health concerns.

### Data Preparation

We'll load the wellness guide as a single document, then split it into smaller chunks using a `RecursiveCharacterTextSplitter` for our vector store. We also keep the raw (unsplit) document for use with the Parent Document Retriever and Semantic Chunker later.

In [4]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

loader = TextLoader("data/HealthWellnessGuide.txt")
raw_docs = loader.load()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
wellness_docs = text_splitter.split_documents(raw_docs)

Let's verify our data was loaded and split correctly!

In [5]:
print(f"Raw documents: {len(raw_docs)}")
print(f"Split chunks: {len(wellness_docs)}")
print(f"\nExample chunk:\n{wellness_docs[0]}")

Raw documents: 1
Split chunks: 45

Example chunk:
page_content='The Personal Wellness Guide
A Comprehensive Resource for Health and Well-being

PART 1: EXERCISE AND MOVEMENT

Chapter 1: Understanding Exercise Basics

Exercise is one of the most important things you can do for your health. Regular physical activity can improve your brain health, help manage weight, reduce the risk of disease, strengthen bones and muscles, and improve your ability to do everyday activities.' metadata={'source': 'data/HealthWellnessGuide.txt'}


## Task 3: Setting up QDrant!

Now that we have our documents, let's create a QDrant VectorStore with the collection name "wellness_guide".

We'll leverage OpenAI's [`text-embedding-3-small`](https://openai.com/blog/new-embedding-models-and-api-updates) because it's a very powerful (and low-cost) embedding model.

> NOTE: We'll be creating additional vectorstores where necessary, but this pattern is still extremely useful.

In [6]:
from langchain_qdrant import QdrantVectorStore
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

vectorstore = QdrantVectorStore.from_documents(
    wellness_docs,
    embeddings,
    location=":memory:",
    collection_name="wellness_guide",
)

## Task 4: Naive RAG Chain

Since we're focusing on the "R" in RAG today - we'll create our Retriever first.

### R - Retrieval

This naive retriever will simply look at each review as a document, and use cosine-similarity to fetch the 10 most relevant documents.

> NOTE: We're choosing `10` as our `k` here to provide enough documents for our reranking process later

In [7]:
naive_retriever = vectorstore.as_retriever(search_kwargs={"k" : 10})

### A - Augmented

We're going to go with a standard prompt for our simple RAG chain today! Nothing fancy here, we want this to mostly be about the Retrieval process.

In [8]:
from langchain_core.prompts import ChatPromptTemplate

RAG_TEMPLATE = """\
You are a helpful and kind assistant. Only use the given context to answer the question.

If you do not know the answer, or are unsure, say you don't know.

Query:
{question}

Context:
{context}
"""

rag_prompt = ChatPromptTemplate.from_template(RAG_TEMPLATE)

### G - Generation

We're going to leverage `gpt-4.1-nano` as our LLM today, as - again - we want this to largely be about the Retrieval process.

In [9]:
from langchain_openai import ChatOpenAI

chat_model = ChatOpenAI(model="gpt-4.1-mini")

### LCEL RAG Chain

We're going to use LCEL to construct our chain.

> NOTE: This chain will be exactly the same across the various examples with the exception of our Retriever!

In [10]:
from langchain_core.runnables import RunnablePassthrough
from operator import itemgetter
from langchain_core.output_parsers import StrOutputParser

naive_retrieval_chain = (
    # INVOKE CHAIN WITH: {"question" : "<<SOME USER QUESTION>>"}
    # "question" : populated by getting the value of the "question" key
    # "context"  : populated by getting the value of the "question" key and chaining it into the base_retriever
    {"context": itemgetter("question") | naive_retriever, "question": itemgetter("question")}
    # "context"  : is assigned to a RunnablePassthrough object (will not be called or considered in the next step)
    #              by getting the value of the "context" key from the previous step
    | RunnablePassthrough.assign(context=itemgetter("context"))
    # "response" : the "context" and "question" values are used to format our prompt object and then piped
    #              into the LLM and stored in a key called "response"
    # "context"  : populated by getting the value of the "context" key from the previous step
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's see how this simple chain does on a few different prompts.

> NOTE: You might think that we've cherry picked prompts that showcase the individual skill of each of the retrieval strategies - you'd be correct!

In [11]:
naive_retrieval_chain.invoke({"question" : "What exercises can help with lower back pain?"})["response"].content

'Exercises that can help with lower back pain include:\n\n- Cat-Cow Stretch: Start on hands and knees, alternate between arching your back up (cat) and letting it sag down (cow). Do 10-15 repetitions.\n- Bird Dog: From hands and knees, extend opposite arm and leg while keeping your core engaged. Hold for 5 seconds, then switch sides. Do 10 repetitions per side.\n- Pelvic Tilts: Lie on your back with knees bent, flatten your back against the floor by tightening abs and tilting pelvis up slightly. Hold for 10 seconds, repeat 8-12 times.\n- Partial Crunches: Lie on your back with knees bent, cross arms over chest, tighten stomach muscles and raise shoulders off floor. Hold briefly, then lower. Do 8-12 repetitions.\n- Knee-to-Chest Stretch: Lie on your back, pull one knee toward your chest while keeping the other foot flat. Hold for 15-30 seconds, then switch legs.\n\nThese gentle stretching and strengthening exercises can help alleviate discomfort and prevent future episodes of lower back

In [18]:
naive_retrieval_chain.invoke({"question" : "How does sleep affect overall health?"})["response"].content

'Sleep has a significant impact on overall health. Adequate and quality sleep—typically 7-9 hours per night—supports physical health by allowing the body to repair tissues, regulate hormones, and strengthen the immune system. It also enhances mental well-being, cognitive functions such as memory and learning, and mood stability. Poor sleep or sleep disorders like insomnia can negatively affect these processes, leading to increased health risks, weakened immunity, and mental health issues. Maintaining good sleep hygiene and creating an optimal sleep environment are essential strategies for promoting overall health.'

In [19]:
naive_retrieval_chain.invoke({"question" : "What are some natural remedies for stress and headaches?"})["response"].content

'Some natural remedies for stress and headaches include:\n\n- Drinking water and staying hydrated\n- Applying cold or warm compresses to the head or neck\n- Resting in a dark, quiet room\n- Gentle massage of the temples and neck\n- Using peppermint or lavender essential oils\n- Maintaining a regular sleep schedule\n- Practicing deep breathing, progressive muscle relaxation, or grounding techniques\n- Taking short walks in nature\n- Listening to calming music\n\nThese approaches can help alleviate stress and headache symptoms naturally.'

Overall, this is not bad! Let's see if we can make it better!

## Task 5: Best-Matching 25 (BM25) Retriever

Taking a step back in time - [BM25](https://www.nowpublishers.com/article/Details/INR-019) is based on [Bag-Of-Words](https://en.wikipedia.org/wiki/Bag-of-words_model) which is a sparse representation of text.

In essence, it's a way to compare how similar two pieces of text are based on the words they both contain.

This retriever is very straightforward to set-up! Let's see it happen down below!


In [12]:
from langchain_community.retrievers import BM25Retriever

bm25_retriever = BM25Retriever.from_documents(wellness_docs)

We'll construct the same chain - only changing the retriever.

In [13]:
bm25_retrieval_chain = (
    {"context": itemgetter("question") | bm25_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's look at the responses!

In [22]:
bm25_retrieval_chain.invoke({"question" : "What exercises can help with lower back pain?"})["response"].content

'To help with lower back pain, some recommended exercises include:\n\n- **Cat-Cow Stretch:** Start on your hands and knees, then alternate between arching your back up (like a cat) and letting it sag down (like a cow). Perform 10-15 repetitions.\n- **Bird Dog:** On your hands and knees, extend the opposite arm and leg while engaging your core, hold for 5 seconds, then switch sides. Do 10 repetitions per side.\n- **Pelvic Tilts:** Lie on your back with knees bent, tighten your abdominal muscles, and tilt your pelvis slightly to flatten your lower back against the floor. Hold for 10 seconds and repeat 8-12 times.\n\nThese exercises can help alleviate discomfort and prevent future episodes of lower back pain.'

In [23]:
bm25_retrieval_chain.invoke({"question" : "How does sleep affect overall health?"})["response"].content

'Sleep significantly impacts overall health in several ways. Adults generally need 7-9 hours of sleep per night, during which the body goes through cycles of approximately 90 minutes, including REM and non-REM stages. During deep sleep (Stage 3), the body repairs and regenerates tissues, supporting physical health. Sleep also facilitates important cognitive functions, such as memory and learning, especially during REM sleep. Additionally, maintaining a regular sleep schedule and creating an optimal sleep environment—such as a cool, dark, and quiet room—can improve sleep quality. Poor sleep or insomnia can negatively affect health by impairing immune function, mental health, and physical recovery.'

In [24]:
bm25_retrieval_chain.invoke({"question" : "What are some natural remedies for stress and headaches?"})["response"].content

'Some natural remedies for stress include practicing relaxation techniques such as deep breathing and meditation. For headaches, staying well-hydrated, managing stress through relaxation, ensuring adequate sleep, and avoiding known triggers like certain foods or eye strain can help reduce their frequency and intensity. Herbal teas like chamomile or valerian root may also promote relaxation and help alleviate headaches related to stress.'

It's not clear that this is better or worse, if only we had a way to test this (SPOILERS: We do, the second half of the notebook will cover this)

### ❓ Question #1:

Give an example query where BM25 is better than embeddings and justify your answer.

##### Answer:

Example query where BM25 can perform better than embeddings: 
"What does Section 1.0 of the ICMR 2025 guideline say about Vitamin D3 dosage?"

The query talks about a specific section, specific document and year and a subject constraint. It is no longer a general medical question. Only exact matching determines correctness, semantic similarity alone is insufficent in this case.

## Task 6: Contextual Compression (Using Reranking)

Contextual Compression is a fairly straightforward idea: We want to "compress" our retrieved context into just the most useful bits.

There are a few ways we can achieve this - but we're going to look at a specific example called reranking.

The basic idea here is this:

- We retrieve lots of documents that are very likely related to our query vector
- We "compress" those documents into a smaller set of *more* related documents using a reranking algorithm.

We'll be leveraging Cohere's Rerank model for our reranker today!

All we need to do is the following:

- Create a basic retriever
- Create a compressor (reranker, in this case)

That's it!

Let's see it in the code below!

In [14]:
from langchain.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain_cohere import CohereRerank

compressor = CohereRerank(model="rerank-v3.5")
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor, base_retriever=naive_retriever
)

Let's create our chain again, and see how this does!

In [15]:
contextual_compression_retrieval_chain = (
    {"context": itemgetter("question") | compression_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

In [87]:
contextual_compression_retrieval_chain.invoke({"question" : "What exercises can help with lower back pain?"})["response"].content

'To help alleviate lower back pain, gentle stretching and strengthening exercises can be beneficial. Some recommended exercises include:\n\n- **Cat-Cow Stretch:** Start on your hands and knees. Alternate between arching your back upward (like a cat) and letting it sag downward (like a cow). Repeat this movement 10-15 times.\n- **Bird Dog:** From your hands and knees, extend your opposite arm and leg simultaneously, keeping your core engaged. Hold this position for about 5 seconds, then switch sides. Aim for 10 repetitions per side.\n- **Pelvic Tilts:** Lie on your back with knees bent. Flatten your lower back against the floor by tightening your abdominal muscles and tilting your pelvis upward slightly. Hold for 10 seconds and repeat 8-12 times.\n\nPlease consult with a healthcare professional before starting any new exercise routine, especially if you have ongoing back issues.'

In [29]:
contextual_compression_retrieval_chain.invoke({"question" : "How does sleep affect overall health?"})["response"].content

'Sleep has a significant impact on overall health. It is essential for physical recovery, as the body repairs tissues and regenerates during deep sleep stages. Sleep also plays a crucial role in mental well-being, helping with memory consolidation and cognitive functions. Additionally, sleep influences the release of hormones that regulate growth and appetite, which can affect overall metabolic health. Adults generally need 7-9 hours of quality sleep per night to support these vital functions and maintain good health.'

In [30]:
contextual_compression_retrieval_chain.invoke({"question" : "What are some natural remedies for stress and headaches?"})["response"].content

'Some natural remedies for stress and headaches include drinking water to stay hydrated, applying cold or warm compresses to the head or neck, resting in a dark, quiet room, gently massaging the temples and neck, and using essential oils like peppermint or lavender. Additionally, practices such as deep breathing, progressive muscle relaxation, grounding techniques, taking short walks in nature, and listening to calming music can help alleviate stress.'

We'll need to rely on something like Ragas to help us get a better sense of how this is performing overall - but it "feels" better!

## Task 7: Multi-Query Retriever

Typically in RAG we have a single query - the one provided by the user.

What if we had....more than one query!

In essence, a Multi-Query Retriever works by:

1. Taking the original user query and creating `n` number of new user queries using an LLM.
2. Retrieving documents for each query.
3. Using all unique retrieved documents as context

So, how is it to set-up? Not bad! Let's see it down below!



In [16]:
from langchain.retrievers.multi_query import MultiQueryRetriever

multi_query_retriever = MultiQueryRetriever.from_llm(
    retriever=naive_retriever, llm=chat_model
) 

In [17]:
multi_query_retrieval_chain = (
    {"context": itemgetter("question") | multi_query_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

In [91]:
multi_query_retrieval_chain.invoke({"question" : "What exercises can help with lower back pain?"})["response"].content

'Exercises that can help with lower back pain include:\n\n- **Cat-Cow Stretch:** Start on your hands and knees, then alternate between arching your back up (cat) and letting it sag down (cow). Perform 10-15 repetitions.\n\n- **Bird Dog:** From hands and knees, extend opposite arm and leg while keeping your core engaged. Hold for 5 seconds, then switch sides. Do 10 repetitions per side.\n\n- **Partial Crunches:** Lie on your back with knees bent, arms crossed over your chest, tighten your stomach muscles, and raise your shoulders off the floor. Do 8-12 repetitions.\n\n- **Knee-to-Chest Stretch:** Lie on your back, pull one knee toward your chest while keeping the other foot flat, hold for 15-30 seconds, then switch legs.\n\n- **Pelvic Tilts:** Lie on your back with knees bent, flatten your back against the floor by tightening your abs and tilting your pelvis up slightly. Hold for 10 seconds, repeat 8-12 times.\n\nThese exercises are gentle and designed to alleviate discomfort and preven

In [34]:
multi_query_retrieval_chain.invoke({"question" : "How does sleep affect overall health?"})["response"].content

'Sleep plays a vital role in overall health by supporting physical recovery, mental well-being, and cognitive functions. During sleep, the body repairs tissues, consolidates memories, and releases hormones that help regulate growth and appetite. Adequate and quality sleep—typically 7-9 hours per night—also contributes to a stronger immune system, better emotional regulation, and overall longevity. Poor sleep or sleep disturbances, such as insomnia, can negatively impact physical health, increase stress, and reduce mental clarity. Therefore, maintaining good sleep hygiene and creating an optimal sleep environment are essential for promoting overall health and well-being.'

In [35]:
multi_query_retrieval_chain.invoke({"question" : "What are some natural remedies for stress and headaches?"})["response"].content

'Some natural remedies for stress and headaches include:\n\n- Deep breathing exercises (inhale for 4 counts, hold for 4, exhale for 4)\n- Progressive muscle relaxation (tensing and releasing muscle groups)\n- Grounding techniques (naming things you see, hear, feel, smell, and taste)\n- Taking short walks, preferably in nature\n- Listening to calming music\n- Drinking water to stay hydrated\n- Applying cold or warm compresses to the head or neck\n- Resting in a dark, quiet room\n- Gentle massage of temples and neck\n- Using essential oils like peppermint or lavender\n\nThese methods can help provide immediate relief and promote overall relaxation.'

### ❓ Question #2:

Explain how generating multiple reformulations of a user query can improve recall.

##### Answer:

Dense vector retrieval maps a query to a single embedding vector. This vector retrieves chunks from one region of vector space. If the user's query under-represents some aspects of his intent then relevant embeddings from other semantic region may never be retrieved leading to lower recall.

Generating multiple reformulations of the same question creates multiple embedding vectors. Each reformulation vector can now match different chunks from different semantic region for the same topic and then take the union of the retrieved results. The union improves the recall because it increases the probability of retrieving all relevant documents.

## Task 8: Parent Document Retriever

A "small-to-big" strategy - the Parent Document Retriever works based on a simple strategy:

1. We split the full document into large "parent" chunks (e.g. 2000 characters).
2. Each parent chunk is further split into smaller "child" chunks (e.g. 400 characters).
3. The child chunks are stored in a VectorStore, while the parent chunks are stored in an in-memory docstore.
4. When we query our Retriever, we do a similarity search comparing our query vector to the child chunks.
5. Instead of returning the child chunks, we return their associated parent chunks.

The basic idea is:

- **Search** for small, focused chunks (better semantic matching)
- **Return** big chunks (richer surrounding context)

The intuition is that we're likely to find the most relevant information by limiting the amount of semantic information encoded in each embedding vector - but we're likely to miss relevant surrounding context if we only use that information.

Let's start by defining our parent and child splitters.

In [18]:
from langchain.retrievers import ParentDocumentRetriever
from langchain.storage import InMemoryStore
from langchain_text_splitters import RecursiveCharacterTextSplitter
from qdrant_client import QdrantClient, models

parent_splitter = RecursiveCharacterTextSplitter(chunk_size=2000, chunk_overlap=200)
child_splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=50)

We'll need to set up a new QDrant vectorstore - and we'll use another useful pattern to do so!

> NOTE: We are manually defining our embedding dimension, you'll need to change this if you're using a different embedding model.

In [19]:
from langchain_qdrant import QdrantVectorStore

client = QdrantClient(location=":memory:")

client.create_collection(
    collection_name="wellness_parent_child",
    vectors_config=models.VectorParams(size=1536, distance=models.Distance.COSINE)
)

parent_document_vectorstore = QdrantVectorStore(
    collection_name="wellness_parent_child", embedding=OpenAIEmbeddings(model="text-embedding-3-small"), client=client
)

Now we can create our `InMemoryStore` that will hold our "parent documents" - and build our retriever!

In [20]:
store = InMemoryStore()

parent_document_retriever = ParentDocumentRetriever(
    vectorstore=parent_document_vectorstore,
    docstore=store,
    child_splitter=child_splitter,
    parent_splitter=parent_splitter,
)

By default, this is empty as we haven't added any documents - let's add some now!

In [21]:
parent_document_retriever.add_documents(raw_docs, ids=None)

We'll create the same chain we did before - but substitute our new `parent_document_retriever`.

In [22]:
parent_document_retrieval_chain = (
    {"context": itemgetter("question") | parent_document_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's give it a whirl!

In [41]:
parent_document_retrieval_chain.invoke({"question" : "What exercises can help with lower back pain?"})["response"].content

"To help with lower back pain, gentle stretching and strengthening exercises are recommended. Some effective exercises include:\n\n- **Cat-Cow Stretch:** On hands and knees, alternate between arching your back up (cat) and letting it sag down (cow). Do 10-15 repetitions.\n- **Bird Dog:** From hands and knees, extend opposite arm and leg while keeping your core engaged. Hold for 5 seconds, then switch sides. Do 10 repetitions per side.\n- **Partial Crunches:** Lie on your back with knees bent, cross arms over your chest, tighten your stomach muscles, and raise your shoulders off the floor. Do 8-12 repetitions.\n- **Knee-to-Chest Stretch:** Lie on your back, pull one knee toward your chest while keeping the other foot flat. Hold for 15-30 seconds, then switch legs.\n- **Pelvic Tilts:** Lie on your back with knees bent, flatten your back against the floor by tightening your abs and tilting your pelvis up slightly. Hold for 10 seconds and repeat 8-12 times.\n\nThese exercises can help alle

In [42]:
parent_document_retrieval_chain.invoke({"question" : "How does sleep affect overall health?"})["response"].content

'Sleep significantly affects overall health in several ways. It is crucial for physical health, mental well-being, and cognitive function. During sleep, the body repairs tissues, consolidates memories, and releases hormones that regulate growth and appetite. Adults generally need 7-9 hours of sleep per night. Good sleep quality supports healthy immune function, maintains mood stability, and enhances concentration and learning. Poor sleep or sleep disturbances can lead to increased risk of health issues such as fatigue, headaches, impaired cognitive function, and emotional stress. Therefore, maintaining healthy sleep habits and hygiene is vital for overall wellness.'

In [43]:
parent_document_retrieval_chain.invoke({"question" : "What are some natural remedies for stress and headaches?"})["response"].content

'Some natural remedies for stress and headaches include practicing deep breathing, engaging in progressive muscle relaxation, doing grounding exercises, taking short walks especially in nature, listening to calming music, and practicing mindfulness or meditation. For headaches specifically, natural remedies include drinking plenty of water to stay hydrated, applying cold or warm compresses to the head or neck, resting in a dark, quiet room, gently massaging the temples and neck, using essential oils like peppermint or lavender, and maintaining a regular sleep schedule.'

Overall, the performance *seems* largely the same. We can leverage a tool like [Ragas]() to more effectively answer the question about the performance.

## Task 9: Ensemble Retriever

In brief, an Ensemble Retriever simply takes 2, or more, retrievers and combines their retrieved documents based on a rank-fusion algorithm.

In this case - we're using the [Reciprocal Rank Fusion](https://plg.uwaterloo.ca/~gvcormac/cormacksigir09-rrf.pdf) algorithm.

Setting it up is as easy as providing a list of our desired retrievers - and the weights for each retriever.

In [23]:
from langchain.retrievers import EnsembleRetriever

retriever_list = [bm25_retriever, naive_retriever, parent_document_retriever]
equal_weighting = [1/len(retriever_list)] * len(retriever_list)

ensemble_retriever = EnsembleRetriever(
    retrievers=retriever_list, weights=equal_weighting
)

We'll pack *all* of these retrievers together in an ensemble.

In [25]:
ensemble_retrieval_chain = (
    {"context": itemgetter("question") | ensemble_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's look at our results!

In [46]:
ensemble_retrieval_chain.invoke({"question" : "What exercises can help with lower back pain?"})["response"].content

'Exercises that can help with lower back pain include:\n\n- **Cat-Cow Stretch:** Start on hands and knees, alternate between arching your back up (like a cat) and letting it sag down (like a cow). Do 10-15 repetitions.\n\n- **Bird Dog:** From hands and knees, extend the opposite arm and leg while keeping your core engaged. Hold for 5 seconds, then switch sides. Do 10 repetitions per side.\n\n- **Pelvic Tilts:** Lie on your back with knees bent, tighten your abs, and tilt your pelvis slightly to flatten your lower back against the floor. Hold for 10 seconds, repeat 8-12 times.\n\n- **Partial Crunches:** Lie on your back with knees bent, cross your arms over your chest, tighten your stomach muscles, and lift your shoulders off the floor briefly. Then lower back down. Do 8-12 repetitions.\n\n- **Knee-to-Chest Stretch:** Lie on your back, pull one knee toward your chest while keeping the other foot flat on the floor. Hold for 15-30 seconds, then switch legs.\n\nRemember to start slowly and

In [47]:
ensemble_retrieval_chain.invoke({"question" : "How does sleep affect overall health?"})["response"].content

'Sleep has a significant impact on overall health. It is essential for physical repair, mental well-being, and cognitive functions such as memory and learning. During sleep, tissues are repaired, hormones that regulate growth and appetite are released, and the brain consolidates memories. Adequate sleep (7-9 hours per night) supports immune function, reduces stress, and boosts mental health. Poor sleep or sleep disturbances like insomnia can negatively affect physical health, mental well-being, and increase vulnerability to illness. Maintaining good sleep hygiene—such as keeping a consistent schedule, creating a comfortable sleep environment, and practicing relaxation techniques—can help promote better sleep quality and, consequently, improve overall health.'

In [48]:
ensemble_retrieval_chain.invoke({"question" : "What are some natural remedies for stress and headaches?"})["response"].content

'Some natural remedies for stress include deep breathing exercises, progressive muscle relaxation, grounding techniques (such as naming things you see, hear, feel, smell, and taste), taking short walks especially in nature, and listening to calming music. \n\nFor headaches, natural remedies include staying well-hydrated by drinking water, applying cold or warm compresses to the head or neck, resting in a dark and quiet room, gentle massage of the temples and neck, using essential oils such as peppermint or lavender, maintaining a regular sleep schedule, and using caffeine in small amounts if appropriate.'

## Task 10: Semantic Chunking

While this is not a retrieval method - it *is* an effective way of increasing retrieval performance on corpora that have clean semantic breaks in them.

Essentially, Semantic Chunking is implemented by:

1. Embedding all sentences in the corpus.
2. Combining or splitting sequences of sentences based on their semantic similarity based on a number of [possible thresholding methods](https://python.langchain.com/docs/how_to/semantic-chunker/):
  - `percentile`
  - `standard_deviation`
  - `interquartile`
  - `gradient`
3. Each sequence of related sentences is kept as a document!

Let's see how to implement this!

We'll use the `percentile` thresholding method for this example which will:

Calculate all distances between sentences, and then break apart sequences of setences that exceed a given percentile among all distances.

In [26]:
from langchain_experimental.text_splitter import SemanticChunker

semantic_chunker = SemanticChunker(
    embeddings,
    breakpoint_threshold_type="percentile"
)

Now we can split our documents.

In [27]:
semantic_documents = semantic_chunker.split_documents(raw_docs)

Let's create a new vector store.

In [28]:
semantic_vectorstore = QdrantVectorStore.from_documents(
    semantic_documents,
    embeddings,
    location=":memory:",
    collection_name="wellness_guide_semantic_chunks"
)

We'll use naive retrieval for this example.

In [29]:
semantic_retriever = semantic_vectorstore.as_retriever(search_kwargs={"k" : 10})

Finally we can create our classic chain!

In [30]:
semantic_retrieval_chain = (
    {"context": itemgetter("question") | semantic_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

And view the results!

In [31]:
semantic_retrieval_chain.invoke({"question" : "What exercises can help with lower back pain?"})["response"].content

'Exercises that can help with lower back pain include:\n\n- Cat-Cow Stretch: Start on hands and knees, alternate between arching your back up (cat) and letting it sag down (cow). Do 10-15 repetitions.\n- Partial Crunches: Lie on your back with knees bent, cross arms over chest, tighten stomach muscles and raise shoulders off the floor. Hold briefly, then lower. Do 8-12 repetitions.\n- Knee-to-Chest Stretch: Lie on your back, pull one knee toward your chest while keeping the other foot flat. Hold for 15-30 seconds, then switch legs.\n- Pelvic Tilts: Lie on your back with knees bent, flatten your back against the floor by tightening abs and tilting pelvis up slightly. Hold for 10 seconds, repeat 8-12 times.\n- Bird Dog: From hands and knees, extend opposite arm and leg while keeping your core engaged. Hold for 5 seconds, then switch sides.\n\nThese gentle stretching and strengthening exercises can help alleviate discomfort and prevent future episodes of lower back pain.'

In [56]:
semantic_retrieval_chain.invoke({"question" : "How does sleep affect overall health?"})["response"].content

'Sleep is essential for overall health because it allows the body to repair tissues, regulate hormones, and support cognitive functions like memory and learning. Adequate sleep (7-9 hours per night) helps maintain physical health, mental well-being, and proper immune function. Poor sleep or disruptions in sleep cycles can lead to issues such as fatigue, headaches, weakened immune response, and impaired mental health. Good sleep hygiene practices and creating an optimal sleep environment are important for ensuring restful sleep and promoting overall wellness.'

In [57]:
semantic_retrieval_chain.invoke({"question" : "What are some natural remedies for stress and headaches?"})["response"].content

'Some natural remedies for stress and headaches include:\n\n- Deep breathing exercises: Inhale for 4 counts, hold for 4, exhale for 4, and hold for 4.\n- Progressive muscle relaxation: Tense and relax muscle groups, such as from toes to head.\n- Grounding techniques: Name 5 things you see, 4 you hear, 3 you feel, 2 you smell, and 1 you taste.\n- Taking short walks, preferably in nature.\n- Listening to calming music.\n- Applying peppermint or lavender essential oils.\n- Staying well-hydrated by drinking water.\n- Resting in a dark, quiet room.\n- Gentle massage of temples and neck.\n\nThese approaches can help reduce tension, promote relaxation, and alleviate headache symptoms naturally.'

### ❓ Question #3:

If sentences are short and highly repetitive (e.g., FAQs), how might semantic chunking behave, and how would you adjust the algorithm?

##### Answer:

When sentences are short and repetitive, semantic chunking can either group unrelated chunks together because they sound similar or split content into too many tiny pieces because each sentence lacks details. This will worsen the retreival. We can adjust the algorithm by
1. Chunking by structure say one QnA per chunk
2. Applying semantic merging only across bloks when similarity is very high
3. Apply only when we have some kind of hybrid structure like say semantics only inside long answers

---

# 🤝 Breakout Room Part #2

### 🏗️ Activity #1:

Your task is to evaluate the various Retriever methods against each other.

You are expected to:

1. Create a "golden dataset"
 - Use Synthetic Data Generation (powered by Ragas, or otherwise) to create this dataset
2. Evaluate each retriever with *retriever specific* Ragas metrics
 - Semantic Chunking is not considered a retriever method and will not be required for marks, but you may find it useful to do a "semantic chunking on" vs. "semantic chunking off" comparison between them
3. Compile these in a list and write a small paragraph about which is best for this particular data and why.

Your analysis should factor in:
  - Cost
  - Latency
  - Performance

> NOTE: This is **NOT** required to be completed in class. Please spend time in your breakout rooms creating a plan before moving on to writing code.

##### HINTS:

- LangSmith provides detailed information about latency and cost.

## SDG using RAGAS

In [32]:
### YOUR CODE HERE

from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings
from ragas.testset import TestsetGenerator
from datasets import Dataset

generator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1"))
generator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings())

generator = TestsetGenerator(llm=generator_llm, embedding_model=generator_embeddings)
dataset = generator.generate_with_langchain_docs(raw_docs, testset_size=12)

dataset.to_pandas()

Applying HeadlinesExtractor:   0%|          | 0/1 [00:00<?, ?it/s]

Applying HeadlineSplitter:   0%|          | 0/1 [00:00<?, ?it/s]

Applying SummaryExtractor:   0%|          | 0/1 [00:00<?, ?it/s]

Applying CustomNodeFilter:   0%|          | 0/4 [00:00<?, ?it/s]

Applying [EmbeddingExtractor, ThemesExtractor, NERExtractor]:   0%|          | 0/9 [00:00<?, ?it/s]

Applying [CosineSimilarityBuilder, OverlapScoreBuilder]:   0%|          | 0/2 [00:00<?, ?it/s]

Generating personas:   0%|          | 0/1 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/1 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/9 [00:00<?, ?it/s]

,user_input,reference_contexts,reference,synthesizer_name
0,Me want know what Chapter 1: Understanding Exe...,[The Personal Wellness Guide A Comprehensive R...,Chapter 1: Understanding Exercise Basics say e...,single_hop_specifc_query_synthesizer
1,What are the main principles outlined in Chapt...,[The Personal Wellness Guide A Comprehensive R...,Chapter 4: Fundamentals of Healthy Eating emph...,single_hop_specifc_query_synthesizer
2,How I do Cat-Cow Stretch for my lower back pai...,[The Personal Wellness Guide A Comprehensive R...,"For Cat-Cow Stretch, start on your hands and k...",single_hop_specifc_query_synthesizer
3,Wut is magnesum gud for sleep?,[PART 3: SLEEP AND RECOVERY Chapter 7: The Sci...,Magnesium supplements are mentioned as a natur...,single_hop_specifc_query_synthesizer
4,Wut is REM sleep and why iz it importent for m...,[PART 3: SLEEP AND RECOVERY Chapter 7: The Sci...,REM (rapid eye movement) sleep is a stage of s...,single_hop_specifc_query_synthesizer
5,"what non-REM sleep do for body, why it matter ...",[PART 3: SLEEP AND RECOVERY Chapter 7: The Sci...,"Sleep go in cycles, got non-REM and REM sleep....",single_hop_specifc_query_synthesizer
6,What are the recommended sources of Vitamin D ...,[PART 5: BUILDING HEALTHY HABITS Chapter 13: T...,"According to the provided wellness guide, Vita...",single_hop_specifc_query_synthesizer
7,Wut are sum practises for digital wellness tha...,[PART 5: BUILDING HEALTHY HABITS Chapter 13: T...,Digital wellness practises include setting spe...,single_hop_specifc_query_synthesizer
8,Which foods are good sources of Vitamin E for ...,[PART 5: BUILDING HEALTHY HABITS Chapter 13: T...,"Nuts, seeds, and spinach are good sources of V...",single_hop_specifc_query_synthesizer


## Retrievar Graph + Retrievar Metrics

In [33]:
import uuid
from langsmith import Client

client = Client()

dataset_name = f"Advanced Retrieval - all retrievers - {uuid.uuid4()}"

langsmith_dataset = client.create_dataset(
    dataset_name=dataset_name,
    description="Advanced Retreival Use Cases"
)

for data_row in dataset.to_pandas().iterrows():
  client.create_example(
      inputs={
          "question": data_row[1]["user_input"]
      },
      outputs={
          "answer": data_row[1]["reference"]
      },
      metadata={
          "context": data_row[1]["reference_contexts"]
      },
      dataset_id=langsmith_dataset.id
  )

retrievers = {
    "chunking_off_naive": naive_retriever,
    "chunking_on_semantic": semantic_retriever,
    "bm25": bm25_retriever,
    "multi_query": multi_query_retriever,
    "parent_document": parent_document_retriever,
    "compression": compression_retriever,
    "ensemble": ensemble_retriever,
}

In [ ]:
# Bharath 2
#build a graph for compression_retriever and calculate the metrics for compression_retriever --> testing evaluation for native_retriever

from langgraph.graph import START, END, StateGraph
from typing_extensions import List, TypedDict
from langchain_core.documents import Document
from ragas import EvaluationDataset
from ragas.llms import LangchainLLMWrapper
from ragas.metrics import LLMContextRecall, ContextEntityRecall, ContextPrecision
from ragas import evaluate as ragas_evaluate, RunConfig
import time
import copy
from langsmith.evaluation import evaluate as langSmith_evaluate

class State(TypedDict):
  question: str
  context: List[Document]
  response: str

def make_retrieve(retriever):
    def retrieve(state):
        retrieved_docs = retriever.invoke(state["question"])
        return {"context": retrieved_docs}
    return retrieve

evaluator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-mini"))
custom_run_config = RunConfig(timeout=360)
all_metrics = {}
all_retrieved_contexts = {}

for name, retriever in retrievers.items():
    # Build graph for this retriever
    retrieve_fn = make_retrieve(retriever)
    graph_builder = StateGraph(State).add_sequence([retrieve_fn])
    graph_builder.add_edge(START, "retrieve")
    graph_builder.add_edge("retrieve", END)
    graph = graph_builder.compile()

    rerank_dataset = copy.deepcopy(dataset)

    for test_row in rerank_dataset:
        response = graph.invoke({"question": test_row.eval_sample.user_input})
        test_row.eval_sample.retrieved_contexts = [context.page_content for context in response["context"]]
        #if isinstance(retriever, ContextualCompressionRetriever):
            #print("Sleep Called")
            #time.sleep(10)  # avoid Cohere API rate limiting for compression_retriever if using trial key

    all_retrieved_contexts[name] = [test_row.eval_sample.retrieved_contexts for test_row in rerank_dataset]

    evaluation_dataset = EvaluationDataset.from_pandas(rerank_dataset.to_pandas())

    result = ragas_evaluate(
        dataset=evaluation_dataset,
        metrics=[LLMContextRecall(), ContextEntityRecall(), ContextPrecision()],
        llm=evaluator_llm,
        run_config=custom_run_config,
    )

    all_metrics[name] = result
    print(f"{name} evaluation metrics:")
    print(result)

    langSmith_evaluate(
        graph,
        data=langsmith_dataset,
        metadata={
            "revision_id": "default_chain_init",
            "retreiver_name": name,
        },
        experiment_prefix=name,
    )

print(len(all_retrieved_contexts))
print("\n--- Summary ---")
for name, result in all_metrics.items():
    print(f"{name}: {result}")

Evaluating:   0%|          | 0/27 [00:00<?, ?it/s]

chunking_off_naive evaluation metrics:
{'context_recall': 0.9259, 'context_entity_recall': 0.4288, 'context_precision': 0.8889}
View the evaluation results for experiment: 'chunking_off_naive-ff0472f8' at:
https://smith.langchain.com/o/8ec58a99-449f-4f06-9fbf-946e52cbfeb5/datasets/55e0c304-497e-41da-a993-5aba1b8ac123/compare?selectedSessions=a30fa39f-1190-4b65-b9eb-4723776dd355




0it [00:00, ?it/s]

Evaluating:   0%|          | 0/27 [00:00<?, ?it/s]

chunking_on_semantic evaluation metrics:
{'context_recall': 0.9722, 'context_entity_recall': 0.3873, 'context_precision': 0.6759}
View the evaluation results for experiment: 'chunking_on_semantic-579121d6' at:
https://smith.langchain.com/o/8ec58a99-449f-4f06-9fbf-946e52cbfeb5/datasets/55e0c304-497e-41da-a993-5aba1b8ac123/compare?selectedSessions=4f8af710-8d86-45f9-b430-d7c353557639




...

0it [00:00, ?it/s]

Evaluating:   0%|          | 0/27 [00:00<?, ?it/s]

ensemble evaluation metrics:
{'context_recall': 0.9630, 'context_entity_recall': 0.5425, 'context_precision': 0.6514}
View the evaluation results for experiment: 'ensemble-86f5c3a8' at:
https://smith.langchain.com/o/8ec58a99-449f-4f06-9fbf-946e52cbfeb5/datasets/55e0c304-497e-41da-a993-5aba1b8ac123/compare?selectedSessions=73618e9c-d0f6-46a7-82a6-ac38783365fd




0it [00:00, ?it/s]

7

--- Summary ---
chunking_off_naive: {'context_recall': 0.9259, 'context_entity_recall': 0.4288, 'context_precision': 0.8889}
chunking_on_semantic: {'context_recall': 0.9722, 'context_entity_recall': 0.3873, 'context_precision': 0.6759}
bm25: {'context_recall': 0.6593, 'context_entity_recall': 0.2658, 'context_precision': 0.7222}
multi_query: {'context_recall': 1.0000, 'context_entity_recall': 0.4325, 'context_precision': 0.8463}
parent_document: {'context_recall': 0.9259, 'context_entity_recall': 0.3958, 'context_precision': 0.8889}
compression: {'context_recall': 0.8981, 'context_entity_recall': 0.3524, 'context_precision': 0.8889}
ensemble: {'context_recall': 0.9630, 'context_entity_recall': 0.5425, 'context_precision': 0.6514}


### Retrieval Metrics Summary

- chunking_off_naive: {'context_recall': 0.9259, 'context_entity_recall': 0.4288, 'context_precision': 0.8889}
- chunking_on_semantic: {'context_recall': 0.9722, 'context_entity_recall': 0.3873, 'context_precision': 0.6759}
- bm25: {'context_recall': 0.6593, 'context_entity_recall': 0.2658, 'context_precision': 0.7222}
- multi_query: {'context_recall': 1.0000, 'context_entity_recall': 0.4325, 'context_precision': 0.8463}
- parent_document: {'context_recall': 0.9259, 'context_entity_recall': 0.3958, 'context_precision': 0.8889}
- compression: {'context_recall': 0.8981, 'context_entity_recall': 0.3524, 'context_precision': 0.8889}
- ensemble: {'context_recall': 0.9630, 'context_entity_recall': 0.5425, 'context_precision': 0.6514}

|Retrievar| context_recall | context_entity_recall | context_precision |Latency from LangSmith (P50) (sec)|
|---------|------------|-----------------|-----------------|-----------------|
|chunking_off_naive|0.9259|0.4288|0.8889|0.244|
|chunking_on_semantic|0.9722|0.3873|0.6759|0.208|
|bm25| 0.6593|0.2658|0.7222|0.002|
|multi_query|1.0000|0.4325|0.8463|1.98|
|parent_document|0.9259|0.3958|0.8889|0.197|
|compression|0.8981|0.3524|0.8889|0.535|
|ensemble|0.9630|0.5425|0.6514|0.47|

- For this evaluation, multi-query retrieval is the strongest overall, it reaches perfect context recall (1.0) and the best balance across metrics (context recall 1.0, context entity recall 0.4325, context precision 0.8463). So for this dataset, rephrasing the question multiple ways and merging results reliably brings in the needed context without as much junk as the ensemble. 
- Ensemble wins on context entity recall (0.5425) and has high context recall (0.9630), but it has the lowest context precision (0.6514), so you pay with more irrelevant chunks. 
- If the priority is precision (fewer false positives for the LLM), chunking_off_naive or parent_document are best (both 0.8889 precision with 0.9259 context recall). 
- BM25 alone is clearly worst (lowest recall and entity recall), so this data benefits from semantic or hybrid strategies.
- In practice: use multi-query when you want high recall and good precision; use chunking_off_naive or parent_document when you care most about precision and a smaller, cleaner context.

## Metrics for Generation

In [41]:
base_df = dataset.to_pandas()

questions = base_df["user_input"].tolist()
reference_contexts = base_df["reference_contexts"].tolist()
reference = base_df["reference"].tolist()

def generate_response(question, retrieved_contexts_list):
    """Generate answer from question + pre-retrieved contexts (no retrieval)."""
    context_str = "\n\n".join(retrieved_contexts_list)
    message = rag_prompt.invoke({"question": question, "context": context_str})
    response = chat_model.invoke(message)
    return response.content if hasattr(response, "content") else str(response)

In [ ]:
# Base data: same questions and references for all generation
from ragas import EvaluationDataset, evaluate
from ragas.metrics import Faithfulness, FactualCorrectness, ResponseRelevancy
import pandas as pd

generation_metrics = [Faithfulness(), FactualCorrectness(), ResponseRelevancy()]
evaluator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-mini"))
custom_run_config = RunConfig(timeout=360)
all_generation_metrics = {}

for name in all_retrieved_contexts:
    retrieved_contexts = all_retrieved_contexts[name]   # reuse; no retrieval again

    # Generate responses using those same retrieved_contexts
    responses = [generate_response(q, ctxs) for q, ctxs in zip(questions, retrieved_contexts)]
    evaluation_df_with_response = pd.DataFrame({
        "user_input": questions,
        "reference_contexts": reference_contexts,
        "reference": reference,
        "retrieved_contexts": retrieved_contexts,
        "response": responses,
    })
    evaluation_dataset_for_generation = EvaluationDataset.from_pandas(evaluation_df_with_response)
    result_new = evaluate(
        dataset=evaluation_dataset_for_generation,
        metrics=generation_metrics,
        llm=evaluator_llm,
        run_config=custom_run_config,
    )
    all_generation_metrics[name] = result_new
    print(f"{name} generation evaluation metrics:")
    print(result_new)


print("\n--- Summary ---")
for name, result in all_generation_metrics.items():
    print(f"{name}: {result}")

Evaluating:   0%|          | 0/27 [00:00<?, ?it/s]

chunking_off_naive generation evaluation metrics:
{'faithfulness': 0.7347, 'factual_correctness': 0.6922, 'answer_relevancy': 0.9561}


Evaluating:   0%|          | 0/27 [00:00<?, ?it/s]

chunking_on_semantic generation evaluation metrics:
{'faithfulness': 0.8811, 'factual_correctness': 0.6889, 'answer_relevancy': 0.8571}


Evaluating:   0%|          | 0/27 [00:00<?, ?it/s]

...

parent_document generation evaluation metrics:
{'faithfulness': 0.6757, 'factual_correctness': 0.6600, 'answer_relevancy': 0.7453}


Evaluating:   0%|          | 0/27 [00:00<?, ?it/s]

compression generation evaluation metrics:
{'faithfulness': 0.7900, 'factual_correctness': 0.8011, 'answer_relevancy': 0.8536}


Evaluating:   0%|          | 0/27 [00:00<?, ?it/s]

ensemble generation evaluation metrics:
{'faithfulness': 0.7259, 'factual_correctness': 0.7356, 'answer_relevancy': 0.8579}

--- Summary ---
chunking_off_naive: {'faithfulness': 0.7347, 'factual_correctness': 0.6922, 'answer_relevancy': 0.9561}
chunking_on_semantic: {'faithfulness': 0.8811, 'factual_correctness': 0.6889, 'answer_relevancy': 0.8571}
bm25: {'faithfulness': 0.7593, 'factual_correctness': 0.5567, 'answer_relevancy': 0.7448}
multi_query: {'faithfulness': 0.7294, 'factual_correctness': 0.7333, 'answer_relevancy': 0.8554}
parent_document: {'faithfulness': 0.6757, 'factual_correctness': 0.6600, 'answer_relevancy': 0.7453}
compression: {'faithfulness': 0.7900, 'factual_correctness': 0.8011, 'answer_relevancy': 0.8536}
ensemble: {'faithfulness': 0.7259, 'factual_correctness': 0.7356, 'answer_relevancy': 0.8579}


### Generation Metrics Summary

- chunking_off_naive: {'faithfulness': 0.7347, 'factual_correctness': 0.6922, 'answer_relevancy': 0.9561}
- chunking_on_semantic: {'faithfulness': 0.8811, 'factual_correctness': 0.6889, 'answer_relevancy': 0.8571}
- bm25: {'faithfulness': 0.7593, 'factual_correctness': 0.5567, 'answer_relevancy': 0.7448}
- multi_query: {'faithfulness': 0.7294, 'factual_correctness': 0.7333, 'answer_relevancy': 0.8554}
- parent_document: {'faithfulness': 0.6757, 'factual_correctness': 0.6600, 'answer_relevancy': 0.7453}
- compression: {'faithfulness': 0.7900, 'factual_correctness': 0.8011, 'answer_relevancy': 0.8536}
- ensemble: {'faithfulness': 0.7259, 'factual_correctness': 0.7356, 'answer_relevancy': 0.8579}

|Retrievar| faithfulness | factual_correctness | answer_relevancy |
|---------|------------|-----------------|-----------------|
|chunking_off_naive|0.7347|0.6922|0.9561|
|chunking_on_semantic|0.8811|0.6889|0.8571|
|bm25|0.7593 |0.5567|0.7448|
|multi_query|0.7294|0.7333|0.8554|
|parent_document|0.6757|0.6600|0.7453|
|compression| 0.7900|0.8011| 0.8536|
|ensemble|0.7259|0.7356|0.8579|

- Chunking_on_semantic gives the highest faithfulness (0.8811)—semantic chunks help the model stay grounded in the retrieved text
- while compression gives the highest factual correctness (0.8011) and strong faithfulness (0.7900), so compressing the context seems to cut noise and support more accurate answers. 
- Chunking_off_naive tops answer relevancy (0.9561) and has solid faithfulness and factual correctness, so for this data a simple chunking strategy still yields very on-topic answers.
- In contrast, multi_query had perfect context recall in retrieval but only moderate faithfulness (0.7294) and factual correctness (0.7333), and parent_document had high retrieval precision but the lowest faithfulness (0.6757) and weaker factual correctness, suggesting that more or broader context can sometimes hurt grounding
- Overall: retrieval quality helps generation (e.g. compression and chunking_on_semantic do well on both), but the kind of retrieval matters, semantic chunking and compression favor faithful, factual answers, while naive chunking favors relevancy; multi-query and parent-document retrieval did not translate into the best generation quality here.